# Airfoil Performance Simulator

This notebook explores the basic aerodynamic behavior of **NACA 4-digit airfoils**. It was built as a computational companion to a subsonic wind-tunnel project, so the goal is to connect aerodynamic theory with quantities that can eventually be compared with experiment.

The model uses **thin-airfoil theory** for lift and pressure trends, along with simplified models for drag and stall. It is useful for learning and comparing trends, but it is **not CFD** and should not be treated as a high-fidelity aerodynamic solver.


## 1. Imports and Air Properties

The simulator uses Numpy for the calculations and Matplotlib for plotting. Air density and dynamic viscosity are set to approximate sea-level values.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

AIR_DENSITY = 1.225       # kg/m^3
AIR_VISCOSITY = 1.81e-5   # Pa*s

## 2. NACA 4-Digit Airfoil Geometry

A 4-digit NACA code describes the basic shape of an airfoil. For example, **NACA 2412** means:

- maximum camber = 2% of the chord
- maximum camber location = 40% of the chord
- maximum thickness = 12% of the chord

The next functions decode the NACA number and generate the upper and lower airfoil surfaces.


In [ ]:
def parse_naca4(code):
    "Convert a NACA 4-digit code into camber, camber location, and thickness."
    code = str(code).strip()

    if len(code) != 4 or not code.isdigit():
        raise ValueError("NACA code must contain exactly four digits.")

    max_camber = int(code[0]) / 100
    camber_location = int(code[1]) / 10
    thickness = int(code[2:]) / 100

    return max_camber, camber_location, thickness

In [ ]:
def naca4_coordinates(code, points=160):
    "Generate upper and lower coordinates for a NACA 4-digit airfoil."
    m, p, t = parse_naca4(code)

    beta = np.linspace(0, np.pi, points)
    x = (1 - np.cos(beta)) / 2

    thickness_distribution = 5 * t * (
        0.2969 * np.sqrt(x)
        - 0.1260 * x
        - 0.3516 * x**2
        + 0.2843 * x**3
        - 0.1015 * x**4
    )

    camber_line = np.zeros_like(x)
    camber_slope = np.zeros_like(x)

    if m != 0 and p != 0:
        front = x < p
        back = ~front

        camber_line[front] = m / p**2 * (
            2 * p * x[front] - x[front]**2
        )

        camber_slope[front] = 2 * m / p**2 * (
            p - x[front]
        )

        camber_line[back] = m / (1 - p)**2 * (
            (1 - 2 * p) + 2 * p * x[back] - x[back]**2
        )

        camber_slope[back] = 2 * m / (1 - p)**2 * (
            p - x[back]
        )

    angle = np.arctan(camber_slope)

    x_upper = x - thickness_distribution * np.sin(angle)
    y_upper = camber_line + thickness_distribution * np.cos(angle)

    x_lower = x + thickness_distribution * np.sin(angle)
    y_lower = camber_line - thickness_distribution * np.cos(angle)

    return x_upper, y_upper, x_lower, y_lower

In [ ]:
def get_camber_slope(code, x):
    "Return the slope of the airfoil's mean camber line."
    m, p, _ = parse_naca4(code)
    x = np.asarray(x)

    if m == 0 or p == 0:
        return np.zeros_like(x, dtype=float)

    return np.where(
        x < p,
        2 * m / p**2 * (p - x),
        2 * m / (1 - p)**2 * (p - x)
    )

## 3. Lift and Zero-Lift Angle

For attached flow, thin-airfoil theory predicts an approximately linear relationship between lift coefficient and angle of attack:

$$C_L \approx 2\pi(\alpha - \alpha_{L=0})$$

where the angles are in radians. A cambered airfoil usually has a negative zero-lift angle, meaning it can produce positive lift even at a geometric angle of attack of zero degrees.

Thin-airfoil theory itself does not predict stall, so the lift function below adds a simple post-stall correction after about 12 degrees. That part is only meant to show the general trend.


In [ ]:
def zero_lift_angle(code):
    "Estimate zero-lift angle using thin-airfoil theory."
    theta = np.linspace(1e-5, np.pi - 1e-5, 4000)
    x = (1 - np.cos(theta)) / 2

    slope = get_camber_slope(code, x)

    integral = np.trapezoid(
        slope * (1 - np.cos(theta)),
        theta
    )

    return np.degrees(integral / np.pi)

In [ ]:
def lift_coefficient(code, angle_of_attack):
    "Estimate lift coefficient using thin-airfoil theory."

    "A simple correction is added after about 12 degrees to show the general effect of stall"

    zero_lift = zero_lift_angle(code)

    cl_linear = 2 * np.pi * np.radians(
        angle_of_attack - zero_lift
    )

    stall_angle = 12

    if -stall_angle <= angle_of_attack <= stall_angle:
        return cl_linear

    if angle_of_attack > stall_angle:
        cl_at_stall = 2 * np.pi * np.radians(
            stall_angle - zero_lift
        )

        extra_angle = angle_of_attack - stall_angle

        return cl_at_stall * np.exp(-0.045 * extra_angle) + 0.15

    cl_at_stall = 2 * np.pi * np.radians(
        -stall_angle - zero_lift
    )

    extra_angle = abs(angle_of_attack + stall_angle)

    return cl_at_stall * np.exp(-0.045 * extra_angle) - 0.15

## 4. Reynolds Number and Drag Estimate

The Reynolds number compares inertial effects with viscous effects in the flow:

$$Re = \frac{\rho Vc}{\mu}$$

where $\rho$ is air density, $V$ is freestream speed, $c$ is chord length, and $\mu$ is dynamic viscosity.

The drag model combines a skin-friction estimate with simple thickness, lift, angle-of-attack, and post-stall terms. Because real airfoil drag depends strongly on boundary-layer behavior and flow separation, these values should be treated as **estimates for trend analysis**.


In [ ]:
def reynolds_number(speed, chord):
    "Calculate Reynolds number."
    return AIR_DENSITY * speed * chord / AIR_VISCOSITY


In [ ]:
def skin_friction_coefficient(reynolds):
    "Estimate skin-friction coefficient."
    if reynolds < 500000:
        return 1.328 / np.sqrt(reynolds)

    turbulent = 0.074 / reynolds**0.2 - 1742 / reynolds
    return max(turbulent, 0.0015)


In [ ]:
def drag_coefficient(code, angle_of_attack, cl, reynolds):
    "Estimate drag coefficient. This is a simplified engineering estimate meant for studying trends."
    _, _, thickness = parse_naca4(code)

    friction = skin_friction_coefficient(reynolds)

    form_factor = 1 + 2 * thickness + 60 * thickness**4

    skin_drag = 2 * friction * form_factor
    lift_drag = 0.008 * cl**2
    angle_drag = 0.002 * abs(angle_of_attack) / 10

    stall_drag = 0

    if abs(angle_of_attack) > 12:
        stall_drag = 0.018 * (
            (abs(angle_of_attack) - 12) / 4
        )**2

    return skin_drag + lift_drag + angle_drag + stall_drag


## 5. Surface Velocity and Pressure Distribution

Thin-airfoil theory can also be used to estimate the circulation distribution around the airfoil. From that, the notebook estimates upper- and lower-surface velocity trends.

The pressure coefficient is calculated from the normalized surface velocity using:

$$C_p = 1 - \left(\frac{V_{surface}}{V_\infty}\right)^2$$

The exact leading edge is skipped because ideal thin-airfoil theory produces a mathematical singularity there.


In [ ]:
def thin_airfoil_coefficients(code, angle_of_attack, terms=8):
    "Calculate Fourier coefficients used by thin-airfoil theory."
    theta = np.linspace(1e-4, np.pi - 1e-4, 4000)
    x = (1 - np.cos(theta)) / 2

    slope = get_camber_slope(code, x)
    alpha = np.radians(angle_of_attack)

    a0 = alpha - np.trapezoid(slope, theta) / np.pi

    coefficients = [a0]

    for n in range(1, terms + 1):
        an = 2 / np.pi * np.trapezoid(
            slope * np.cos(n * theta),
            theta
        )

        coefficients.append(an)

    return coefficients

In [ ]:
def surface_pressure_data(code, angle_of_attack, points=180):
    "Estimate surface velocity and pressure coefficient. This uses thin-airfoil theory, so it is meant to show trends instead of exact CFD-quality results."

    coefficients = thin_airfoil_coefficients(
        code,
        angle_of_attack
    )

    theta = np.linspace(0.08, np.pi - 0.03, points)
    x = (1 - np.cos(theta)) / 2

    a0 = coefficients[0]

    circulation = (
        a0 * (1 + np.cos(theta)) / np.sin(theta)
    )

    for n, an in enumerate(coefficients[1:], start=1):
        circulation += an * np.sin(n * theta)

    velocity_upper = 1 + circulation
    velocity_lower = 1 - circulation

    velocity_upper = np.clip(velocity_upper, 0.05, 4.0)
    velocity_lower = np.clip(velocity_lower, 0.05, 4.0)

    cp_upper = 1 - velocity_upper**2
    cp_lower = 1 - velocity_lower**2

    return (
        x,
        velocity_upper,
        velocity_lower,
        cp_upper,
        cp_lower
    )

## 6. Input and Result Formatting

These helper functions handle user input and print the main aerodynamic results in a readable format.


In [ ]:
def ask(prompt, default, cast=float):
    "Ask the user for a value while allowing a default."
    response = input(f"{prompt} [{default}]: ").strip()

    if response == "":
        return cast(default)

    return cast(response)

In [ ]:
def print_results(
    naca,
    angle_of_attack,
    speed,
    chord,
    span,
    reynolds,
    cl,
    cd,
    lift,
    drag
):
    "Print the main aerodynamic results."
    print("\n" + "=" * 50)
    print("AIRFOIL PERFORMANCE RESULTS")
    print("=" * 50)

    print(f"NACA airfoil:          {naca}")
    print(f"Angle of attack:       {angle_of_attack:.2f} deg")
    print(f"Air speed:             {speed:.2f} m/s")
    print(f"Chord length:          {chord:.3f} m")
    print(f"Reference span:        {span:.3f} m")
    print(f"Reynolds number:       {reynolds:,.0f}")
    print(f"Lift coefficient:      {cl:.3f}")
    print(f"Drag coefficient:      {cd:.4f}")
    print(f"Lift force:            {lift:.2f} N")
    print(f"Drag force:            {drag:.2f} N")

    print("=" * 50)

## 7. Plotting Functions

The following functions visualize the airfoil geometry and the main aerodynamic trends predicted by the model.


In [ ]:
def plot_airfoil(naca):
    "Plot the airfoil geometry."
    x_upper, y_upper, x_lower, y_lower = naca4_coordinates(naca)

    plt.figure(figsize=(9, 4))

    plt.plot(x_upper, y_upper, label="Upper surface")
    plt.plot(x_lower, y_lower, label="Lower surface")

    plt.axhline(0, linewidth=0.7)

    plt.xlabel("x / chord")
    plt.ylabel("y / chord")
    plt.title(f"NACA {naca} Airfoil")

    plt.axis("equal")
    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()

In [ ]:
def plot_lift_curve(naca):
    "Plot lift coefficient over a range of angles of attack."
    angles = np.linspace(-12, 20, 150)

    lift_values = []

    for angle in angles:
        lift_values.append(
            lift_coefficient(naca, angle)
        )

    plt.figure(figsize=(8, 5))

    plt.plot(angles, lift_values)

    plt.xlabel("Angle of attack (deg)")
    plt.ylabel("Lift coefficient, Cl")
    plt.title(f"Lift Curve - NACA {naca}")

    plt.grid(alpha=0.3)
    plt.tight_layout()



In [ ]:
def plot_pressure_distribution(naca, angle_of_attack):
    "Plot estimated pressure coefficient over the chord."
    (
        x,
        velocity_upper,
        velocity_lower,
        cp_upper,
        cp_lower
    ) = surface_pressure_data(naca, angle_of_attack)

    plt.figure(figsize=(8, 5))

    plt.plot(x, cp_upper, label="Upper surface")
    plt.plot(x, cp_lower, label="Lower surface")

    plt.gca().invert_yaxis()

    plt.xlabel("x / chord")
    plt.ylabel("Pressure coefficient, Cp")
    plt.title(
        f"Pressure Distribution at {angle_of_attack:.1f} deg"
    )

    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()

In [ ]:
def plot_velocity_distribution(naca, angle_of_attack):
    "Plot estimated surface velocity over the chord."
    (
        x,
        velocity_upper,
        velocity_lower,
        cp_upper,
        cp_lower
    ) = surface_pressure_data(naca, angle_of_attack)

    plt.figure(figsize=(8, 5))

    plt.plot(x, velocity_upper, label="Upper surface")
    plt.plot(x, velocity_lower, label="Lower surface")

    plt.xlabel("x / chord")
    plt.ylabel("Velocity / freestream velocity")
    plt.title(
        f"Surface Velocity at {angle_of_attack:.1f} deg"
    )

    plt.grid(alpha=0.3)
    plt.legend()
    plt.tight_layout()


In [ ]:
def plot_reynolds_effect(naca, angle_of_attack, chord):
    "Show how the drag estimate changes with Reynolds number."
    speeds = np.linspace(5, 50, 100)

    reynolds_values = []
    drag_values = []

    cl = lift_coefficient(naca, angle_of_attack)

    for speed in speeds:
        re = reynolds_number(speed, chord)
        cd = drag_coefficient(
            naca,
            angle_of_attack,
            cl,
            re
        )

        reynolds_values.append(re)
        drag_values.append(cd)

    plt.figure(figsize=(8, 5))

    plt.plot(reynolds_values, drag_values)

    plt.xlabel("Reynolds number")
    plt.ylabel("Estimated drag coefficient, Cd")
    plt.title("Reynolds Number Effect on Drag")

    plt.grid(alpha=0.3)
    plt.tight_layout()



## 8. Run the Simulator

The main function brings all of the calculations together. It asks for:

- a NACA 4-digit airfoil
- angle of attack
- air speed
- chord length
- reference span

Press **Enter** at any prompt to use the default value. The default case is NACA 2412 at five degrees, 20 m/s, with a 0.30 m chord and 0.50 m reference span.


In [ ]:
def main():
    print("Airfoil Performance Simulator")
    print("Press Enter to use the default values.\n")

    naca = ask(
        "NACA 4-digit airfoil",
        "2412",
        str
    ).replace("NACA", "").strip()

    angle_of_attack = ask(
        "Angle of attack in degrees",
        5.0
    )

    speed = ask(
        "Air speed in m/s",
        20.0
    )

    chord = ask(
        "Chord length in meters",
        0.30
    )

    span = ask(
        "Reference span in meters",
        0.50
    )

    if len(naca) != 4 or not naca.isdigit():
        raise ValueError(
            "Enter a 4-digit NACA airfoil such as 2412."
        )

    if speed <= 0 or chord <= 0 or span <= 0:
        raise ValueError(
            "Speed, chord, and span must be positive."
        )

    reynolds = reynolds_number(
        speed,
        chord
    )

    cl = lift_coefficient(
        naca,
        angle_of_attack
    )

    cd = drag_coefficient(
        naca,
        angle_of_attack,
        cl,
        reynolds
    )

    wing_area = chord * span

    dynamic_pressure = (
        0.5 * AIR_DENSITY * speed**2
    )

    lift = (
        dynamic_pressure
        * wing_area
        * cl
    )

    drag = (
        dynamic_pressure
        * wing_area
        * cd
    )

    print_results(
        naca,
        angle_of_attack,
        speed,
        chord,
        span,
        reynolds,
        cl,
        cd,
        lift,
        drag
    )

    plot_airfoil(naca)

    plot_lift_curve(naca)

    plot_pressure_distribution(
        naca,
        angle_of_attack
    )

    plot_velocity_distribution(
        naca,
        angle_of_attack
    )

    plot_reynolds_effect(
        naca,
        angle_of_attack,
        chord
    )

    plt.show()


### Start the simulation

Run the cell below and enter your test conditions when prompted. The notebook will print the aerodynamic summary and generate the geometry, lift, pressure, velocity, and Reynolds-number plots.


In [ ]:
main()

## Model Limitations and Possible Next Steps

This notebook is an educational model rather than a full aerodynamic solver. It does not resolve turbulence, detailed flow separation, three-dimensional wingtip effects, or compressibility.
